# Download sample episodes from [XDOF/ABC-130k](https://huggingface.co/datasets/XDOF/ABC-130k)

Interactive `notebook` version of [`abc_130k/download.py`](../abc_130k/download.py).

ABC-130k is a large open-source bimanual robot-teleoperation dataset collected on
two-arm YAM stations. Each episode is distributed as a single MCAP file. 

The dataset is **gated** — make sure you're authenticated first. `huggingface_hub`
picks up your token automatically, whether from `hf auth login` or `$HF_TOKEN`.

> **Kernel:** run this in the pixi env. Click **Select Kernel** (top-right) →
> **Python Environments…** → `.pixi/envs/abc-130k/bin/python` (run `pixi install -e abc-130k`
> first if that env is missing).

In [ ]:
# import packages
from __future__ import annotations

from pathlib import Path

import pandas as pd
from huggingface_hub import HfApi, hf_hub_download
from IPython.display import HTML

from rrd_datasets_common.paths import dataset_data_dir

## [Dataset Structure](https://huggingface.co/datasets/XDOF/ABC-130k#dataset-structure)
```shell
XDOF/ABC-130k/
├── README.md
├── data/
│   ├── train/                        # training split
│   │   └── <task_name>/
│   │       ├── episode_XXXX/
│   │       │   ├── episode.mcap       # trajectory: joint state, gripper, video, calibration, task name
│   │       │   └── annotation.mcap    # subtask labels — annotated episodes only
│   │       └── ...
│   └── val/                           # validation split
│       └── <task_name>/
│           └── ...
└── meta/
    ├── train_report.txt               # note: actual file names are different from their description
    └── val_report.txt                 #      (includes  task list, trajectory counts, hours) 
```

## Download meta data

It provides available task names and the number of episodes per task. 

In [ ]:
REPO_ID = "XDOF/ABC-130k"

# The shared data root (<workspace>/data/ABC-130k) — the same place `pixi run abc-download`
# puts the sample episode. rrd_datasets_common.paths resolves the workspace root from a
# notebook kernel too (nearest ancestor holding pixi.toml).
LOCAL_DIR = dataset_data_dir("ABC-130k")

api = HfApi()

# The `meta/` folder holds small dataset-level report files (train_report.txt,
# val_report.txt). We fetch just those fast, avoiding `snapshot_download(allow_patterns="meta/*")` which
# enumerates the *entire* repo tree (>1 TB, 130k episodes) before filtering.
meta_files = [f.path for f in api.list_repo_tree(REPO_ID, path_in_repo="meta", repo_type="dataset")]
for f in meta_files:
    hf_hub_download(
        repo_id=REPO_ID,
        repo_type="dataset",
        filename=f,
        local_dir=str(LOCAL_DIR),
    )
print(f"Downloaded {len(meta_files)} meta files to {LOCAL_DIR / 'meta'}: {meta_files}")


def load_task_report(path: Path) -> pd.DataFrame:
    """Parse an ABC-130k `*_report.txt` into a task-stats DataFrame."""
    lines = path.read_text().splitlines()
    seps = [i for i, ln in enumerate(lines) if set(ln.strip()) == {"-"}]
    rows = [ln.rsplit(maxsplit=3) for ln in lines[seps[0] + 1 : seps[1]] if ln.strip()]
    df = pd.DataFrame(rows, columns=["Task Name", "Episodes", "Annotated", "Hours"])
    return df.astype({"Episodes": int, "Annotated": int, "Hours": float})


report = load_task_report(LOCAL_DIR / "meta" / "train_report.txt")

# Render inside a fixed-height, scrollable box so all rows are reachable without the
# default pandas "..." truncation flooding the notebook — scroll to see the rest.
HTML(f'<div style="max-height: 400px; overflow: auto;">{report.to_html()}</div>')

## Download specified number of episodes for a specific task
Specify the following variables in the following cell.
```
TASK = 
NUM_EPISODES = 
```

Some interesting episodes including edge cases from `train` are:

| Example | Features |
| --- | --- |
| `remove_the_shorts_from_the_hanger/episode_0132ab84-8dd2-4fb7-ade9-12b68989720d` | single-top-camera · 640×480 D405 @30Hz · no annotation · **short duration (17.5 s)** |
| `set_up_the_chess_pieces_on_the_board/episode_001ecd72-c651-4389-a275-0fba7d0f438a` | single-top-camera · 640×480 OAK-1-W-97 @30Hz · has annotation · **long duration (320.1 s)** |
| `fold_the_paper_box/episode_0444c81-7761-4e9c-a28f-6b1da1992eb2` | **dual-top-camera** · **1920×1200** ZED_X @30Hz · no annotation · **high-rate proprio** |
| `screw_on_the_bottle_caps/episode_011d8c5b-b147-4293-b05c-a2567f939e66` | single-top-camera · 640×480 D405 @60Hz · **high-rate proprio** |
| `clip_the_underwear_to_the_hanger/episode_0893975b-d8a5-4e92-9291-1f37c416b25a` | single-top-camera · **848×480** D405 @60Hz · has annotation |
| `fold_the_napkin_into_a_case_and_place_the_utensils_inside/episode_00cb478a-aa77-4e5f-ac5a-e53f62914d3f` | single-top-camera · **1280×720** D405 @30Hz · has annotation |
| `build_the_wood_block_tower/episode_00784ada-77e4-4fdc-b26a-d76a1f24cab0` | single-top-camera · 640×480 D405 @30Hz · no annotation · **missing `/left-wrist-camera-info`** |
| `remove_the_keys_from_the_keyring/episode_a8a5e956-840e-45e6-a949-02b211218877` | single-top-camera · **800×608** decxin @30Hz · no annotation · **high-rate proprio** · **resolution mismatch with `camera-info`** |
| `place_the_utensils_on_the_paper_napkin_and_roll_it_up/episode_011640fb-8b08-4100-81b3-6e10f44708d3` | single-top-camera · 1280×1024 decxin @30Hz · no annotation · **identity intrinsics** |

In [ ]:
from abc_130k.episode_index import discover_episodes

####
TASK = "fold_a_paper_plane_with_a_square_sheet_of_paper_fold_left_side_to_right_side"
NUM_EPISODES = 10
####


def download_task_episodes(task: str, n: int, split: str = "train") -> list[str]:
    """
    Download the first `n` episodes of `task` into the shared data root (data/ABC-130k).

    Episode paths come from the launcher's cached listing (`.cache/hf_files.json.gz`, pinned to
    the dataset revision).
    The first run takes a while (a few minutes). After that it's cached and runs faster.

    Episodes are UUID-named, so "first n" means the n lowest uuids.
    Annotated episodes also carry an annotation.mcap; both files come along.
    Task names are the `Task Name` column of the report table above (e.g. "connect_and_route_the_hose").
    """
    # The trailing slash anchors the match, so a task whose name prefixes another can't pull it in.
    episodes = discover_episodes(REPO_ID, f"data/{split}/{task}/")[:n]
    files = []
    for episode in episodes:
        files.append(f"{episode.episode_dir}/episode.mcap")
        if episode.has_annotation:
            files.append(f"{episode.episode_dir}/annotation.mcap")

    return [
        hf_hub_download(
            repo_id=REPO_ID,
            repo_type="dataset",
            filename=filename,
            local_dir=str(LOCAL_DIR),
        )
        for filename in files
    ]


# Heads-up: episodes are ~50-140 MB each, so 10 episodes would be the order of ~1 GB.
paths = download_task_episodes(TASK, NUM_EPISODES)
print(f"Downloaded {len(paths)} files to {LOCAL_DIR}")